# nb02 — Step 5: Positive Pair Redefinition
## Breaking Class-Level Collapse via Trajectory-Similarity Pairs

---

**Purpose:** nb01 established that CTLS with class-label positive pairs produces class-level collapse:
within-image augmentation similarity (0.930) ≈ same-class pair similarity (0.937), meaning `z` encodes
only class identity. The meta-encoder learns `z ≈ h₈` — dominated by the last layer — because the
self-reinforcing cycle pushes both backbone and meta-encoder toward a late-layer-dominant solution.

**Step 5 hypothesis:** Replacing class-label positive pairs with pairs defined by *per-layer trajectory
similarity* breaks this collapse. Two images that process similarly across all layers (not just the last
one) are a positive pair — even if they belong to different classes. This forces the meta-encoder to
track multi-layer circuit structure rather than class identity.

**Training protocol (two-phase):**
1. **Phase 1 (30 epochs):** Standard CTLS with class-label pairs — establishes basic circuit structure.
2. **Precompute:** Extract uniform-mean trajectory embeddings from the Phase 1 model; find top-k
   nearest neighbors per image across the full training set (ignoring class labels).
3. **Phase 2 (70 epochs):** Switch to trajectory-similarity pairs; continue end-to-end training.

**Non-triviality tests (what we must show to claim the step works):**
- T1: Class-level collapse broken — within-image sim > same-class sim (ordering restored)
- T2: `z` adds over `h₈` — `ρ(z-sim, labels) > ρ(h₈-sim, labels)` with measurable gap
- T3: Early-layer gap — per-layer gap table shows layers 1–6 contributing (vs near-zero in nb01)
- T4: Cross-class positive pairs — a meaningful fraction of traj-sim pairs cross class boundaries
- T5: Per-layer ρ profile flatter than nb01 (not dominated by layer 8)

All results compared directly against nb01 Option A checkpoint.

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import subprocess, sys

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Clone repo if running in Colab
if IN_COLAB:
    import os
    REPO = '/content/trainable-circuits'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', 'https://github.com/YOUR_USERNAME/trainable-circuits', REPO], check=True)
    sys.path.insert(0, REPO)
else:
    import os
    # Adjust if running locally
    REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    sys.path.insert(0, REPO)

# Install extras
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=False)

print('Setup complete.')

In [ ]:
import os, random, copy, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm

# Project imports
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from models.soft_mask import SoftMask
from losses.simclr import NTXentLoss

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/trainable_circuits'   # adjust if needed
DATA_DIR      = '/content/data/cifar10'
NB01_CKPT     = f'{DRIVE_ROOT}/experiments/unified_a/best.pt'  # nb01 Option A
NB02_PHASE1   = f'{DRIVE_ROOT}/experiments/step5/phase1.pt'
NB02_BEST     = f'{DRIVE_ROOT}/experiments/step5/best.pt'

Path(f'{DRIVE_ROOT}/experiments/step5').mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

## Section 1 — Architecture & Training Utilities

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


# ── Paired dataset (class-label) ───────────────────────────────────────────────
class PairedCIFAR10(torch.utils.data.Dataset):
    """Returns (img1, img2, label) where img1/img2 are same-class."""
    def __init__(self, root, train=True, transform=None, download=True):
        self.cifar = CIFAR10(root=root, train=train, transform=transform, download=download)
        self.class_to_idx = defaultdict(list)
        for i, lbl in enumerate(self.cifar.targets):
            self.class_to_idx[lbl].append(i)
    def __len__(self): return len(self.cifar)
    def __getitem__(self, idx):
        img1, label = self.cifar[idx]
        j = random.choice(self.class_to_idx[label])
        img2, _ = self.cifar[j]
        return img1, img2, label


# ── Model factory ──────────────────────────────────────────────────────────────
def build_model(device):
    cfg = dict(arch='resnet18', num_classes=10, pretrained=False)
    soft_mask = SoftMask(init_temperature=1.0)
    backbone = CTLSBackbone(**cfg, soft_mask=soft_mask).to(device)
    meta_enc = MetaEncoder(
        layer_dims=backbone.layer_dims,
        embedding_dim=64,
        encoder_type='weighted_sum',
        projection_dim=128,
    ).to(device)
    return backbone, meta_enc, soft_mask


# ── Lambda & tau schedules ─────────────────────────────────────────────────────
def get_lambda(epoch, warmup=10, final=1.0):
    if epoch < warmup:
        return final * epoch / warmup
    return final

def get_tau(epoch, init=1.0, final=0.1, anneal=80):
    t = min(epoch / anneal, 1.0)
    return init + (final - init) * t


# ── Train/eval helpers ─────────────────────────────────────────────────────────
def train_epoch(backbone, meta_enc, loader, optimizer, infonce_loss, lam, device):
    backbone.train(); meta_enc.train()
    tot_loss = tot_task = tot_info = 0.0
    for x1, x2, labels in tqdm(loader, leave=False):
        optimizer.zero_grad()
        x1, x2, labels = x1.to(device), x2.to(device), labels.to(device)
        B = x1.shape[0]
        x_cat = torch.cat([x1, x2], dim=0)
        logits_cat, traj_cat = backbone(x_cat)
        logits1, logits2 = logits_cat[:B], logits_cat[B:]
        traj1 = [h[:B] for h in traj_cat]
        traj2 = [h[B:] for h in traj_cat]
        task = (F.cross_entropy(logits1, labels) + F.cross_entropy(logits2, labels)) / 2.0
        z1, z2 = meta_enc(traj1), meta_enc(traj2)
        info = infonce_loss(z1, z2)
        loss = task + lam * info
        loss.backward()
        nn.utils.clip_grad_norm_(
            list(backbone.parameters()) + list(meta_enc.parameters()), 1.0)
        optimizer.step()
        tot_loss += loss.item(); tot_task += task.item(); tot_info += info.item()
    n = len(loader)
    return tot_loss/n, tot_task/n, tot_info/n


@torch.no_grad()
def eval_acc(backbone, root, device, batch_size=256):
    backbone.eval()
    ds = CIFAR10(root=root, train=False, transform=val_tf, download=True)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    correct = total = 0
    for x, y in dl:
        logits, _ = backbone(x.to(device))
        correct += (logits.argmax(-1) == y.to(device)).sum().item()
        total += y.size(0)
    return correct / total


def save_ckpt(path, backbone, meta_enc, optimizer, epoch, val_acc):
    torch.save({
        'epoch': epoch, 'val_acc': val_acc,
        'backbone': backbone.state_dict(),
        'meta_enc': meta_enc.state_dict(),
        'optimizer': optimizer.state_dict(),
    }, path)


def load_ckpt(path, backbone, meta_enc, optimizer=None, device=None):
    ckpt = torch.load(path, map_location=device or 'cpu')
    backbone.load_state_dict(ckpt['backbone'])
    meta_enc.load_state_dict(ckpt['meta_enc'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    print(f'Loaded {path}  (epoch={ckpt["epoch"]}, val_acc={ckpt["val_acc"]:.4f})')
    return ckpt['epoch']

print('Utilities ready.')

## Section 2 — Phase 1: Class-Label CTLS (30 epochs)

Train CTLS with standard class-label positive pairs to establish baseline circuit structure.
After Phase 1 we extract trajectory embeddings and build the trajectory-similarity pair index.

Set `SKIP_PHASE1 = True` if you already have a Phase 1 checkpoint at `NB02_PHASE1`.

In [ ]:
SKIP_PHASE1 = os.path.exists(NB02_PHASE1)
PHASE1_EPOCHS = 30

backbone_nb02, meta_enc_nb02, soft_mask_nb02 = build_model(DEVICE)
params = list(backbone_nb02.parameters()) + list(meta_enc_nb02.parameters())
optimizer_nb02 = AdamW(params, lr=1e-3, weight_decay=1e-4)
lr_sched_nb02  = CosineAnnealingLR(optimizer_nb02, T_max=100, eta_min=1e-5)
infonce        = NTXentLoss(temperature=0.07)

if SKIP_PHASE1:
    start_epoch = load_ckpt(NB02_PHASE1, backbone_nb02, meta_enc_nb02, optimizer_nb02, DEVICE)
    # Fast-forward lr scheduler
    for _ in range(start_epoch + 1):
        lr_sched_nb02.step()
    print(f'Skipping Phase 1 — resumed from epoch {start_epoch}.')
else:
    train_ds_p1 = PairedCIFAR10(DATA_DIR, train=True, transform=train_tf)
    train_dl_p1 = DataLoader(train_ds_p1, batch_size=256, shuffle=True, num_workers=2)

    best_p1_acc = 0.0
    print(f'Phase 1: {PHASE1_EPOCHS} epochs with class-label pairs')
    for epoch in range(PHASE1_EPOCHS):
        lam = get_lambda(epoch)
        tau = get_tau(epoch)
        soft_mask_nb02.set_temperature(tau)
        loss, task, info = train_epoch(
            backbone_nb02, meta_enc_nb02, train_dl_p1, optimizer_nb02, infonce, lam, DEVICE)
        lr_sched_nb02.step()
        acc = eval_acc(backbone_nb02, DATA_DIR, DEVICE)
        print(f'  Ep {epoch+1:3d} | loss={loss:.4f} task={task:.4f} info={info:.4f} | '
              f'val_acc={acc:.4f} | λ={lam:.3f} τ={tau:.3f}')
        if acc > best_p1_acc:
            best_p1_acc = acc
    save_ckpt(NB02_PHASE1, backbone_nb02, meta_enc_nb02, optimizer_nb02, PHASE1_EPOCHS-1, best_p1_acc)
    print(f'Phase 1 complete. Best val acc = {best_p1_acc:.4f}')

## Section 3 — Precompute Trajectory-Similarity Pair Index

**Method:** Extract per-layer projections `p_l(x)` using the Phase 1 meta-encoder's projectors,
compute `m(x) = normalize(mean_l(p_l(x)))` — a uniform-weighted trajectory embedding that gives
equal weight to all 8 layers. This contrasts with the Phase 1 `z`, which (due to class-label
training) is dominated by layers 7–8.

For each training image, we find its **top-k cosine neighbors** in this uniform trajectory space.
These neighbors — regardless of class label — become the positive pair candidates for Phase 2.

In [ ]:
@torch.no_grad()
def extract_uniform_trajectory_embeddings(backbone, meta_enc, root, device, batch_size=256):
    """Extract m(x) = normalize(mean_l(p_l(x))) for all training images.
    
    Uses the meta-encoder's projectors (shared with nb01 design) but applies
    *uniform* layer weighting — not the depth ramp from weighted_sum.
    This gives each layer equal influence, directly contrasting with nb01's z.
    """
    backbone.eval(); meta_enc.eval()
    # No augmentation — we want stable trajectory embeddings
    ds = CIFAR10(root=root, train=True, transform=val_tf, download=True)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2)
    labels_all = []
    embs_all   = []
    for x, y in tqdm(dl, desc='Extracting trajectory embeddings'):
        x = x.to(device)
        _, traj = backbone(x)
        # Project each layer to 128-D
        projected = [proj(h) for proj, h in zip(meta_enc.projectors, traj)]  # L × [B, 128]
        # Uniform mean across layers, then L2-normalize
        mean_emb = torch.stack(projected, dim=1).mean(dim=1)   # [B, 128]
        mean_emb = F.normalize(mean_emb, dim=-1)               # [B, 128]
        embs_all.append(mean_emb.cpu())
        labels_all.append(y)
    embs   = torch.cat(embs_all, dim=0)    # [N, 128]
    labels = torch.cat(labels_all, dim=0)  # [N]
    print(f'Extracted embeddings: {embs.shape}  (L2-norm mean={embs.norm(dim=-1).mean():.3f})')
    return embs, labels


traj_embs, train_labels = extract_uniform_trajectory_embeddings(
    backbone_nb02, meta_enc_nb02, DATA_DIR, DEVICE)

In [ ]:
@torch.no_grad()
def build_topk_index(embs, k=20, batch_size=512, device='cpu'):
    """Batched exact cosine NN search. Returns [N, k] index tensor.
    
    Memory per batch: B × N × 4 bytes = 512 × 50000 × 4 = 102 MB on GPU.
    Reduce batch_size if OOM.
    """
    N = embs.shape[0]
    embs_dev = embs.to(device)
    top_k_idx = []
    for start in tqdm(range(0, N, batch_size), desc='Building NN index'):
        end = min(start + batch_size, N)
        batch = embs_dev[start:end]           # [B, D]
        sims  = torch.mm(batch, embs_dev.t()) # [B, N]
        # Exclude self
        for i in range(end - start):
            sims[i, start + i] = -2.0
        _, idx = sims.topk(k, dim=1, largest=True, sorted=True)
        top_k_idx.append(idx.cpu())
    return torch.cat(top_k_idx, dim=0)  # [N, k]


K_NEIGHBORS = 20
topk_idx = build_topk_index(traj_embs, k=K_NEIGHBORS, batch_size=512, device=DEVICE)
print(f'Top-k index shape: {topk_idx.shape}')

### 3a — Cross-Class Analysis of Proposed Positive Pairs (Test T4)

Before training Phase 2, we measure what fraction of trajectory-similarity positive pairs cross class
boundaries. This is our first non-triviality check: if trajectory-similarity pairs are
indistinguishable from class-label pairs (i.e., 0% cross-class), then Step 5 adds nothing.

In [ ]:
# T4: Cross-class rate in top-k traj-sim pairs
labels_np = train_labels.numpy()   # [N]
topk_np   = topk_idx.numpy()       # [N, k]

cross_class_per_image = []
for i in range(len(labels_np)):
    neighbor_labels = labels_np[topk_np[i]]          # [k]
    cross = (neighbor_labels != labels_np[i]).mean()  # fraction cross-class
    cross_class_per_image.append(cross)

cross_rate = np.mean(cross_class_per_image)
cross_std  = np.std(cross_class_per_image)

print(f'Cross-class rate in top-{K_NEIGHBORS} traj-sim pairs:')
print(f'  Mean: {cross_rate:.3f} ± {cross_std:.3f}')
print(f'  Expected if purely class-label: ~0.0')
print(f'  Expected if random:             ~0.9 (9/10 classes)')
print()

# Distribution of cross-class rates per image
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(cross_class_per_image, bins=21, color='steelblue', edgecolor='white')
ax.axvline(cross_rate, color='red', linestyle='--', label=f'Mean={cross_rate:.3f}')
ax.set_xlabel('Fraction of top-20 neighbors from different class')
ax.set_ylabel('# images')
ax.set_title('T4: Cross-class rate in traj-sim positive pairs (Phase 1 model)')
ax.legend()
plt.tight_layout(); plt.show()

# Also: what % of images have at least one cross-class neighbor?
any_cross = np.mean([c > 0 for c in cross_class_per_image])
print(f'  Images with ≥1 cross-class neighbor: {any_cross:.3f}')
print()
if cross_rate > 0.05:
    print('✓ T4 SETUP: Cross-class pairs present — trajectory-sim differs from class-label pairs.')
else:
    print('✗ T4 WARNING: Very few cross-class pairs. Trajectory embeddings may still be class-dominated.')
    print('  This is expected if Phase 1 class-label training already collapsed z.')
    print('  Proceeding with Phase 2 — trajectory-sim pairs may still reshape early-layer organization.')

## Section 4 — Trajectory-Similarity Pair Dataset

In [ ]:
class TrajSimPairedCIFAR10(torch.utils.data.Dataset):
    """Paired dataset using precomputed trajectory-similarity neighbors.
    
    For each image i, its positive is sampled uniformly from its top-k
    trajectory-similar neighbors — regardless of class label.
    The task-loss label is still image i's true class label (for accuracy).
    """
    def __init__(self, root, topk_idx, transform=None, download=True):
        self.cifar    = CIFAR10(root=root, train=True, transform=transform, download=download)
        self.topk_idx = topk_idx  # [N, k] LongTensor or ndarray

    def __len__(self): return len(self.cifar)

    def __getitem__(self, idx):
        img1, label = self.cifar[idx]
        # Sample one of the top-k traj-sim neighbors
        neighbors = self.topk_idx[idx]
        j = int(random.choice(neighbors.tolist() if hasattr(neighbors, 'tolist') else list(neighbors)))
        img2, _ = self.cifar[j]
        return img1, img2, label


# Verify dataset
ds_test = TrajSimPairedCIFAR10(DATA_DIR, topk_idx, transform=train_tf)
x1, x2, lbl = ds_test[0]
print(f'TrajSimPairedCIFAR10: x1={x1.shape}, x2={x2.shape}, label={lbl}')
print(f'  Top-k neighbors for image 0: {topk_idx[0][:5].tolist()} ...')
print(f'  Their classes: {[labels_np[j] for j in topk_idx[0][:5].tolist()]} vs image class {labels_np[0]}')

## Section 5 — Phase 2: Trajectory-Similarity CTLS (70 epochs)

Continue training from the Phase 1 checkpoint, switching positive pairs to trajectory-similarity
neighbors. The rest of the training config (λ schedule, τ schedule, optimizer) is unchanged.

Set `SKIP_PHASE2 = True` if you already have a Phase 2 checkpoint at `NB02_BEST`.

In [ ]:
SKIP_PHASE2   = os.path.exists(NB02_BEST)
PHASE2_EPOCHS = 70
TOTAL_EPOCHS  = PHASE1_EPOCHS + PHASE2_EPOCHS  # 100

if SKIP_PHASE2:
    load_ckpt(NB02_BEST, backbone_nb02, meta_enc_nb02, device=DEVICE)
    print('Skipping Phase 2 — loaded nb02 best checkpoint.')
else:
    train_ds_p2 = TrajSimPairedCIFAR10(DATA_DIR, topk_idx, transform=train_tf)
    train_dl_p2 = DataLoader(train_ds_p2, batch_size=256, shuffle=True, num_workers=2)

    best_nb02_acc = 0.0
    print(f'Phase 2: {PHASE2_EPOCHS} epochs with trajectory-similarity pairs (k={K_NEIGHBORS})')
    for ep_offset in range(PHASE2_EPOCHS):
        epoch = PHASE1_EPOCHS + ep_offset
        lam = get_lambda(epoch)   # λ=1.0 throughout Phase 2
        tau = get_tau(epoch)
        soft_mask_nb02.set_temperature(tau)
        loss, task, info = train_epoch(
            backbone_nb02, meta_enc_nb02, train_dl_p2, optimizer_nb02, infonce, lam, DEVICE)
        lr_sched_nb02.step()
        acc = eval_acc(backbone_nb02, DATA_DIR, DEVICE)
        print(f'  Ep {epoch+1:3d} | loss={loss:.4f} task={task:.4f} info={info:.4f} | '
              f'val_acc={acc:.4f} | λ={lam:.3f} τ={tau:.3f}')
        if acc > best_nb02_acc:
            best_nb02_acc = acc
            save_ckpt(NB02_BEST, backbone_nb02, meta_enc_nb02, optimizer_nb02, epoch, acc)

    print(f'Phase 2 complete. Best val acc = {best_nb02_acc:.4f}')
    # Reload best
    load_ckpt(NB02_BEST, backbone_nb02, meta_enc_nb02, device=DEVICE)

## Section 6 — Load nb01 Option A Baseline

nb01 Option A (class-label CTLS, 100 epochs) is the reference model. All Step 5 claims are
relative to it. Load from the nb01 experiment checkpoint.

In [ ]:
# Build nb01 model (same architecture)
backbone_nb01, meta_enc_nb01, _ = build_model(DEVICE)

if os.path.exists(NB01_CKPT):
    # nb01 checkpoint uses different key names (from UnifiedTrainer)
    ckpt = torch.load(NB01_CKPT, map_location=DEVICE)
    # Try both naming conventions
    bb_key = 'backbone' if 'backbone' in ckpt else 'backbone_state'
    me_key = 'meta_enc' if 'meta_enc' in ckpt else 'meta_encoder_state'
    backbone_nb01.load_state_dict(ckpt[bb_key])
    meta_enc_nb01.load_state_dict(ckpt[me_key])
    nb01_acc = ckpt.get('val_acc', float('nan'))
    print(f'nb01 loaded: val_acc={nb01_acc:.4f}')
else:
    print(f'WARNING: nb01 checkpoint not found at {NB01_CKPT}')
    print('Results will show nb02 only. Provide nb01 checkpoint for comparison.')
    nb01_acc = float('nan')

backbone_nb01.eval(); meta_enc_nb01.eval()
backbone_nb02.eval(); meta_enc_nb02.eval()
print('Both models loaded and set to eval mode.')

## Section 7 — Validation Battery

Replicate the full nb01 validation suite on both models. All pair sets and eval code match
nb01 exactly so results are directly comparable.

**Pair sets:**
- `same_pairs`: 500 pairs where both images are from the same class
- `diff_pairs`: 500 pairs where images are from different classes
- `all_pairs` : the 1000 combined pairs used for ρ computation

In [ ]:
# ── Validation dataset & pair construction ─────────────────────────────────────
val_ds_raw = CIFAR10(root=DATA_DIR, train=False, transform=val_tf, download=True)
val_labels = np.array(val_ds_raw.targets)
val_class_to_idx = defaultdict(list)
for i, lbl in enumerate(val_labels):
    val_class_to_idx[lbl].append(i)

rng = np.random.default_rng(SEED)
N_PAIRS_EACH = 500

same_pairs, diff_pairs = [], []
classes = list(range(10))

for _ in range(N_PAIRS_EACH):
    c = rng.choice(classes)
    a, b = rng.choice(val_class_to_idx[c], size=2, replace=False)
    same_pairs.append((int(a), int(b), c, c))

for _ in range(N_PAIRS_EACH):
    c1, c2 = rng.choice(classes, size=2, replace=False)
    a = rng.choice(val_class_to_idx[c1])
    b = rng.choice(val_class_to_idx[c2])
    diff_pairs.append((int(a), int(b), c1, c2))

all_pairs = same_pairs + diff_pairs
pair_labels_binary = np.array([1]*N_PAIRS_EACH + [0]*N_PAIRS_EACH, dtype=np.float32)

print(f'Validation pairs: {N_PAIRS_EACH} same-class + {N_PAIRS_EACH} diff-class = {len(all_pairs)} total')


@torch.no_grad()
def compute_layer_sims(backbone, pairs, dataset, device, batch_size=64):
    """Per-layer cosine similarities for all pairs. Returns [N_pairs, L] array."""
    backbone.eval()
    all_sims = []
    for start in range(0, len(pairs), batch_size):
        chunk = pairs[start:start+batch_size]
        for a_idx, b_idx, *_ in chunk:
            xa = dataset[a_idx][0].unsqueeze(0).to(device)
            xb = dataset[b_idx][0].unsqueeze(0).to(device)
            _, ta = backbone(xa)
            _, tb = backbone(xb)
            sims = [F.cosine_similarity(ha, hb).item() for ha, hb in zip(ta, tb)]
            all_sims.append(sims)
    return np.array(all_sims, dtype=np.float32)  # [N, L]


@torch.no_grad()
def compute_z_sims(backbone, meta_enc, pairs, dataset, device):
    """Circuit embedding cosine similarities for all pairs. Returns [N_pairs] array."""
    backbone.eval(); meta_enc.eval()
    sims = []
    for a_idx, b_idx, *_ in pairs:
        xa = dataset[a_idx][0].unsqueeze(0).to(device)
        xb = dataset[b_idx][0].unsqueeze(0).to(device)
        _, ta = backbone(xa)
        _, tb = backbone(xb)
        za = meta_enc(ta)
        zb = meta_enc(tb)
        sims.append(F.cosine_similarity(za, zb).item())
    return np.array(sims, dtype=np.float32)  # [N]


print('Computing per-layer sims (nb01)...')
layer_sims_nb01 = compute_layer_sims(backbone_nb01, all_pairs, val_ds_raw, DEVICE)
print('Computing per-layer sims (nb02)...')
layer_sims_nb02 = compute_layer_sims(backbone_nb02, all_pairs, val_ds_raw, DEVICE)

print('Computing z sims (nb01)...')
z_sims_nb01 = compute_z_sims(backbone_nb01, meta_enc_nb01, all_pairs, val_ds_raw, DEVICE)
print('Computing z sims (nb02)...')
z_sims_nb02 = compute_z_sims(backbone_nb02, meta_enc_nb02, all_pairs, val_ds_raw, DEVICE)

# Mean trajectory similarity (proxy ground truth — same as nb01)
mean_traj_sims_nb01 = layer_sims_nb01.mean(axis=1)  # [N]
mean_traj_sims_nb02 = layer_sims_nb02.mean(axis=1)

L = layer_sims_nb01.shape[1]
print(f'Done. L={L} layers, {len(all_pairs)} pairs.')

### 7a — Accuracy Comparison

In [ ]:
acc_nb02 = eval_acc(backbone_nb02, DATA_DIR, DEVICE)
print('Val accuracy comparison')
print(f'  nb01 (class-label CTLS): {nb01_acc:.4f}')
print(f'  nb02 (traj-sim   CTLS):  {acc_nb02:.4f}')
print()
delta = acc_nb02 - nb01_acc
print(f'  Δ = {delta:+.4f}  ({'improved' if delta >= 0 else 'degraded'})')
print()
print('Note: accuracy is a secondary metric here. The primary goal is circuit structure.') 
print('A small accuracy drop (< 1%) is acceptable if interpretability metrics improve.')

### 7b — Proxy ρ: z-similarity vs. per-layer trajectory similarity

In [ ]:
rho_nb01, p_nb01 = stats.spearmanr(z_sims_nb01, mean_traj_sims_nb01)
rho_nb02, p_nb02 = stats.spearmanr(z_sims_nb02, mean_traj_sims_nb02)

print('7b. Proxy ρ  =  Spearman ρ(z-sim, mean-traj-sim)')
print(f'  nb01 (class-label): ρ = {rho_nb01:.3f}  p = {p_nb01:.2e}')
print(f'  nb02 (traj-sim):    ρ = {rho_nb02:.3f}  p = {p_nb02:.2e}')
print()
print(f'  Δρ = {rho_nb02 - rho_nb01:+.3f}')
print()
print('Interpretation: nb02 ρ reflects how well z tracks the uniform-mean trajectory')
print('across all 8 layers. Higher ρ means z has learned to represent multi-layer structure.')

### 7c — Per-Layer ρ Profile: which layers does z track? (Test T5)

In [ ]:
perlayer_rho_nb01 = [stats.spearmanr(z_sims_nb01, layer_sims_nb01[:, l])[0] for l in range(L)]
perlayer_rho_nb02 = [stats.spearmanr(z_sims_nb02, layer_sims_nb02[:, l])[0] for l in range(L)]

print('7c. Per-layer ρ(z-sim, sim_l)')
print(f'  Layer | nb01 ρ | nb02 ρ | Δ')
print(f'  ------+--------+--------+------')
for l in range(L):
    d = perlayer_rho_nb02[l] - perlayer_rho_nb01[l]
    print(f'  {l+1:5d} | {perlayer_rho_nb01[l]:6.3f} | {perlayer_rho_nb02[l]:6.3f} | {d:+.3f}')

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(1, L+1)
ax.plot(x, perlayer_rho_nb01, 'o--', color='gray',      label='nb01 (class-label)', linewidth=1.5)
ax.plot(x, perlayer_rho_nb02, 's-',  color='steelblue', label='nb02 (traj-sim)',    linewidth=2)
ax.axhline(0, color='black', linewidth=0.5, linestyle=':')
ax.set_xlabel('Layer'); ax.set_ylabel('Spearman ρ(z-sim, sim_l)')
ax.set_title('T5: Per-layer ρ profile — nb01 vs nb02\n(Flatter = more uniform multi-layer tracking)')
ax.set_xticks(x); ax.legend(); ax.set_ylim(-0.1, 1.0)
plt.tight_layout(); plt.show()

# T5 check
early_rho_nb01 = np.mean(perlayer_rho_nb01[:4])
early_rho_nb02 = np.mean(perlayer_rho_nb02[:4])
late_rho_nb01  = np.mean(perlayer_rho_nb01[4:])
late_rho_nb02  = np.mean(perlayer_rho_nb02[4:])
print()
print(f'T5 summary:')
print(f'  Early layers (1-4): nb01={early_rho_nb01:.3f}  nb02={early_rho_nb02:.3f}  Δ={early_rho_nb02-early_rho_nb01:+.3f}')
print(f'  Late  layers (5-8): nb01={late_rho_nb01:.3f}  nb02={late_rho_nb02:.3f}  Δ={late_rho_nb02-late_rho_nb01:+.3f}')
t5_pass = (early_rho_nb02 > early_rho_nb01 + 0.05)
print(f'  T5: Early-layer ρ improved by >0.05? {"PASS" if t5_pass else "FAIL"}')

### 7d — Per-Layer Gap Table (Test T3)

In [ ]:
# Same-class vs diff-class per-layer cosine similarity gap
same_layer_nb01 = layer_sims_nb01[:N_PAIRS_EACH]   # [500, L]
diff_layer_nb01 = layer_sims_nb01[N_PAIRS_EACH:]   # [500, L]
same_layer_nb02 = layer_sims_nb02[:N_PAIRS_EACH]
diff_layer_nb02 = layer_sims_nb02[N_PAIRS_EACH:]

gap_nb01 = same_layer_nb01.mean(axis=0) - diff_layer_nb01.mean(axis=0)  # [L]
gap_nb02 = same_layer_nb02.mean(axis=0) - diff_layer_nb02.mean(axis=0)

print('7d. Per-layer gap: mean(same-class sim_l) - mean(diff-class sim_l)')
print(f'  Layer | same(nb01) | diff(nb01) | gap(nb01) | same(nb02) | diff(nb02) | gap(nb02)')
print(f'  ------+------------+------------+-----------+------------+------------+-----------')
for l in range(L):
    print(f'  {l+1:5d} | {same_layer_nb01[:,l].mean():10.3f} | {diff_layer_nb01[:,l].mean():10.3f} | '
          f'{gap_nb01[l]:9.3f} | {same_layer_nb02[:,l].mean():10.3f} | {diff_layer_nb02[:,l].mean():10.3f} | {gap_nb02[l]:9.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
x = np.arange(1, L+1)
for ax, gap, title in zip(axes, [gap_nb01, gap_nb02], ['nb01 (class-label)', 'nb02 (traj-sim)']):
    colors = ['#d62728' if g < 0.05 else '#2ca02c' for g in gap]
    ax.bar(x, gap, color=colors)
    ax.axhline(0.05, color='gray', linestyle='--', linewidth=1, label='threshold=0.05')
    ax.set_xlabel('Layer'); ax.set_ylabel('Same - Diff cosine gap')
    ax.set_title(f'Per-layer gap: {title}')
    ax.set_xticks(x); ax.legend()
plt.suptitle('T3: Early-layer gaps > 0 in nb02? (green = gap≥0.05)', y=1.02)
plt.tight_layout(); plt.show()

# T3 check: how many early layers (1-4) show gap > 0.05?
n_early_gaps_nb01 = sum(gap_nb01[:4] > 0.05)
n_early_gaps_nb02 = sum(gap_nb02[:4] > 0.05)
print(f'\nT3: Early layers (1-4) with gap>0.05:')
print(f'  nb01: {n_early_gaps_nb01}/4')
print(f'  nb02: {n_early_gaps_nb02}/4')
t3_pass = (n_early_gaps_nb02 > n_early_gaps_nb01)
print(f'  T3: nb02 has more early-layer gaps than nb01? {"PASS" if t3_pass else "FAIL"}')

### 7e — Baseline Comparison: z vs h₈ vs h₁ (Test T2)

In [ ]:
# h₁ and h₈ cosine similarity
h1_sims_nb01 = layer_sims_nb01[:, 0]   # layer 1
h8_sims_nb01 = layer_sims_nb01[:, -1]  # layer 8
h1_sims_nb02 = layer_sims_nb02[:, 0]
h8_sims_nb02 = layer_sims_nb02[:, -1]

baselines = [
    ('z (nb01)',  z_sims_nb01,         'nb01'),
    ('h₈ (nb01)', h8_sims_nb01,        'nb01'),
    ('h₁ (nb01)', h1_sims_nb01,        'nb01'),
    ('z (nb02)',  z_sims_nb02,         'nb02'),
    ('h₈ (nb02)', h8_sims_nb02,        'nb02'),
    ('h₁ (nb02)', h1_sims_nb02,        'nb02'),
]

print('7e. Baseline comparison: Spearman ρ vs. binary pair label (1=same-class, 0=diff-class)')
print(f'  Metric        | ρ vs labels | p-value')
print(f'  --------------+-------------+--------')
rho_results = {}
for name, sims, model in baselines:
    r, p = stats.spearmanr(sims, pair_labels_binary)
    rho_results[name] = r
    print(f'  {name:14s} | {r:11.3f} | {p:.2e}')

print()
# T2 check: z adds over h8 in nb02?
z_advantage_nb01 = rho_results['z (nb01)']  - rho_results['h₈ (nb01)']
z_advantage_nb02 = rho_results['z (nb02)']  - rho_results['h₈ (nb02)']
print(f'T2: z advantage over h₈ (Δρ vs labels):')
print(f'  nb01: z-ρ - h₈-ρ = {z_advantage_nb01:+.3f}  (should be ≈0 — class-level collapse)')
print(f'  nb02: z-ρ - h₈-ρ = {z_advantage_nb02:+.3f}  (should be >0 — z adds multi-layer info)')
t2_pass = (z_advantage_nb02 > z_advantage_nb01 + 0.02)
print(f'  T2: z adds over h₈ in nb02 more than nb01? {"PASS" if t2_pass else "FAIL"}')

### 7f — Augmentation Invariance (Test T1: Class-Level Collapse Broken?)

**nb01 finding:** within-image augmentation similarity (0.930) ≈ same-class similarity (0.937).
This proves collapse: z cannot distinguish instance identity within a class.

**Step 5 prediction:** within-image sim > same-class sim (ordering restored).
z should be more similar to other views of *the same image* than to arbitrary same-class images,
because trajectory-similarity pairs preserve instance-level circuit identity.

In [ ]:
augment_tf = transforms.Compose([
    transforms.RandomResizedCrop(32, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])


@torch.no_grad()
def augmentation_invariance(backbone, meta_enc, dataset, device,
                             n_images=200, k_aug=5):
    """For n_images, compute mean z-sim across K augmented view pairs (within-image).
    
    Returns within_sims [n_images], same_class_sims [n_images], diff_class_sims [n_images].
    """
    backbone.eval(); meta_enc.eval()
    raw_ds = dataset  # already has .cifar attribute

    val_ds_aug = CIFAR10(root=DATA_DIR, train=False, transform=augment_tf, download=False)
    val_ds_cls = CIFAR10(root=DATA_DIR, train=False, transform=val_tf,    download=False)
    targets    = np.array(val_ds_cls.targets)

    class_to_idx_val = defaultdict(list)
    for i, lbl in enumerate(targets):
        class_to_idx_val[lbl].append(i)

    indices = rng.choice(len(val_ds_cls), n_images, replace=False)
    within_sims, same_sims, diff_sims = [], [], []

    for idx in tqdm(indices, desc='Augmentation invariance'):
        # Within-image: K augmented views of the same image
        views = [val_ds_aug[idx][0].unsqueeze(0).to(device) for _ in range(k_aug)]
        zs = [meta_enc(backbone(v)[1]) for v in views]
        w_sims = []
        for i in range(k_aug):
            for j in range(i+1, k_aug):
                w_sims.append(F.cosine_similarity(zs[i], zs[j]).item())
        within_sims.append(np.mean(w_sims))

        # Same-class: K other images from the same class
        lbl = int(targets[idx])
        pool = [i for i in class_to_idx_val[lbl] if i != idx]
        partners = rng.choice(pool, min(k_aug, len(pool)), replace=False)
        s_sims = []
        for j in partners:
            xa = val_ds_cls[idx][0].unsqueeze(0).to(device)
            xb = val_ds_cls[j][0].unsqueeze(0).to(device)
            za = meta_enc(backbone(xa)[1])
            zb = meta_enc(backbone(xb)[1])
            s_sims.append(F.cosine_similarity(za, zb).item())
        same_sims.append(np.mean(s_sims))

        # Diff-class: K images from a different class
        other_cls = rng.choice([c for c in range(10) if c != lbl])
        others = rng.choice(class_to_idx_val[other_cls], min(k_aug, len(class_to_idx_val[other_cls])), replace=False)
        d_sims = []
        for j in others:
            xa = val_ds_cls[idx][0].unsqueeze(0).to(device)
            xb = val_ds_cls[j][0].unsqueeze(0).to(device)
            za = meta_enc(backbone(xa)[1])
            zb = meta_enc(backbone(xb)[1])
            d_sims.append(F.cosine_similarity(za, zb).item())
        diff_sims.append(np.mean(d_sims))

    return np.array(within_sims), np.array(same_sims), np.array(diff_sims)


print('Computing augmentation invariance for nb01...')
wi_nb01, sa_nb01, di_nb01 = augmentation_invariance(
    backbone_nb01, meta_enc_nb01, val_ds_raw, DEVICE, n_images=200)
print('Computing augmentation invariance for nb02...')
wi_nb02, sa_nb02, di_nb02 = augmentation_invariance(
    backbone_nb02, meta_enc_nb02, val_ds_raw, DEVICE, n_images=200)

print()
print('7f. Augmentation invariance (T1: class-level collapse test)')
print(f'  Condition        |   nb01   |   nb02')
print(f'  -----------------+----------+--------')
print(f'  Within-image     | {wi_nb01.mean():.3f}    | {wi_nb02.mean():.3f}')
print(f'  Same-class       | {sa_nb01.mean():.3f}    | {sa_nb02.mean():.3f}')
print(f'  Diff-class       | {di_nb01.mean():.3f}    | {di_nb02.mean():.3f}')
print()
print(f'  nb01 ordering: within={wi_nb01.mean():.3f} {'≈' if abs(wi_nb01.mean()-sa_nb01.mean())<0.01 else '>'} same={sa_nb01.mean():.3f}')
print(f'  nb02 ordering: within={wi_nb02.mean():.3f} {'>' if wi_nb02.mean() > sa_nb02.mean() else '≈'} same={sa_nb02.mean():.3f}')
gap_nb01_collapse = wi_nb01.mean() - sa_nb01.mean()
gap_nb02_collapse = wi_nb02.mean() - sa_nb02.mean()
t1_pass = (gap_nb02_collapse > gap_nb01_collapse + 0.01)
print(f'  T1: within-image gap over same-class: nb01={gap_nb01_collapse:+.3f}  nb02={gap_nb02_collapse:+.3f}')
print(f'  T1: Gap improved in nb02 (collapse reduced)? {"PASS" if t1_pass else "FAIL"}')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (wi, sa, di), title in zip(
        axes,
        [(wi_nb01, sa_nb01, di_nb01), (wi_nb02, sa_nb02, di_nb02)],
        ['nb01 (class-label)', 'nb02 (traj-sim)']):
    means = [wi.mean(), sa.mean(), di.mean()]
    sems  = [wi.std()/math.sqrt(len(wi)), sa.std()/math.sqrt(len(sa)), di.std()/math.sqrt(len(di))]
    bars  = ax.bar(['Within-image', 'Same-class', 'Diff-class'], means,
                   yerr=sems, capsize=5, color=['#2ca02c', '#1f77b4', '#d62728'])
    ax.set_ylim(0, 1.1); ax.set_ylabel('Mean z cosine similarity')
    ax.set_title(f'{title}\nT1: within > same > diff?')
    # Mark ordering status
    ok = means[0] > means[1] > means[2]
    ax.text(0.5, 0.95, '✓ Ordered' if ok else '✗ Collapsed',
            transform=ax.transAxes, ha='center', fontsize=12,
            color='green' if ok else 'red', fontweight='bold')
plt.suptitle('T1: Augmentation Invariance — Class-Level Collapse Test', fontsize=13)
plt.tight_layout(); plt.show()

## Section 8 — Bootstrap Confidence Intervals

2000-resample bootstrap 95% CIs on all key Spearman ρ values.

In [ ]:
def bootstrap_spearmanr(x, y, n_boot=2000, ci=0.95, seed=0):
    rng_b = np.random.default_rng(seed)
    n = len(x)
    rhos = [stats.spearmanr(
                x[idx := rng_b.integers(0, n, n)], y[idx]
            )[0] for _ in range(n_boot)]
    alpha = 1 - ci
    lo, hi = np.percentile(rhos, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(rhos), lo, hi


print('Bootstrap 95% CIs (2000 resamples)')
print(f'  Metric                          |   ρ   |  95% CI')
print(f'  --------------------------------+-------+--------------')

boot_metrics = [
    # (label, x, y)
    ('ρ(z-sim, traj-sim) nb01',       z_sims_nb01, mean_traj_sims_nb01),
    ('ρ(z-sim, traj-sim) nb02',       z_sims_nb02, mean_traj_sims_nb02),
    ('ρ(z-sim, labels)   nb01',       z_sims_nb01, pair_labels_binary),
    ('ρ(z-sim, labels)   nb02',       z_sims_nb02, pair_labels_binary),
    ('ρ(h₈-sim, labels)  nb01',       h8_sims_nb01, pair_labels_binary),
    ('ρ(h₈-sim, labels)  nb02',       h8_sims_nb02, pair_labels_binary),
]

boot_results = {}
for label, x, y in boot_metrics:
    mu, lo, hi = bootstrap_spearmanr(x, y)
    boot_results[label] = (mu, lo, hi)
    print(f'  {label:32s} | {mu:5.3f} | [{lo:.3f}, {hi:.3f}]')

print()
# Per-layer ρ with CIs
print(f'  Per-layer ρ(z-sim, sim_l) with 95% CI:')
print(f'  Layer | nb01 ρ [CI]              | nb02 ρ [CI]')
print(f'  ------+-------------------------+-------------------------')
for l in range(L):
    mu01, lo01, hi01 = bootstrap_spearmanr(z_sims_nb01, layer_sims_nb01[:, l])
    mu02, lo02, hi02 = bootstrap_spearmanr(z_sims_nb02, layer_sims_nb02[:, l])
    print(f'  {l+1:5d} | {mu01:.3f} [{lo01:.3f},{hi01:.3f}]  | {mu02:.3f} [{lo02:.3f},{hi02:.3f}]')

### 8a — Statistical Test: Δρ Significantly Positive?

We use a permutation test to check whether the difference in ρ between nb02 and nb01 is
statistically significant, not just a random fluctuation on the same pair set.

In [ ]:
def permutation_test_delta_rho(x1, y, x2, n_perm=2000, seed=1):
    """Test H0: ρ(x1,y) == ρ(x2,y) via label permutation.
    
    Returns observed delta, p-value (one-sided: delta > 0).
    """
    rng_p = np.random.default_rng(seed)
    obs_rho1 = stats.spearmanr(x1, y)[0]
    obs_rho2 = stats.spearmanr(x2, y)[0]
    obs_delta = obs_rho2 - obs_rho1
    
    n = len(y)
    null_deltas = []
    for _ in range(n_perm):
        perm = rng_p.permutation(n)
        r1 = stats.spearmanr(x1[perm], y)[0]
        r2 = stats.spearmanr(x2[perm], y)[0]
        null_deltas.append(r2 - r1)
    
    p_val = np.mean(np.array(null_deltas) >= obs_delta)
    return obs_delta, p_val, np.array(null_deltas)


print('Permutation test: H0: ρ(z_nb02, traj-sim) == ρ(z_nb01, traj-sim)')
delta_traj, p_traj, null_traj = permutation_test_delta_rho(
    z_sims_nb01, mean_traj_sims_nb02,   # Use nb02's own ground truth
    z_sims_nb02, mean_traj_sims_nb02)
print(f'  Observed Δρ = {delta_traj:+.3f}  p = {p_traj:.4f}  (one-sided: nb02 > nb01)')
print()

print('Permutation test: H0: z_nb02 labels ρ == h₈_nb02 labels ρ (T2 significance)')
delta_t2, p_t2, null_t2 = permutation_test_delta_rho(
    h8_sims_nb02, pair_labels_binary,
    z_sims_nb02, pair_labels_binary)
print(f'  Observed Δρ(z-h₈) = {delta_t2:+.3f}  p = {p_t2:.4f}  (one-sided: z > h₈)')
print()

# Plot null distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, (null, obs, p, title) in zip(axes, [
    (null_traj, delta_traj, p_traj, 'Δρ(z-sim, traj-sim): nb02 - nb01'),
    (null_t2,   delta_t2,   p_t2,   'Δρ vs labels: z - h₈ (nb02)'),
]):
    ax.hist(null, bins=40, color='lightgray', edgecolor='gray')
    ax.axvline(obs, color='red', linewidth=2, label=f'Observed Δ={obs:+.3f}')
    ax.axvline(0, color='black', linewidth=0.8, linestyle=':')
    ax.set_title(f'{title}\np={p:.4f}')
    ax.set_xlabel('Δρ'); ax.set_ylabel('Count')
    ax.legend()
plt.suptitle('Permutation null distributions (H0: Δρ = 0)', fontsize=12)
plt.tight_layout(); plt.show()

## Section 9 — UMAP Visualization

Side-by-side UMAP of z-space for nb01 vs nb02, colored by class label.

**What to look for:**
- nb01: clean class clusters, no visible within-class structure (collapse)
- nb02: class clusters still present (for classification) but with visible within-class sub-structure
  (instance-level circuit identity preserved). Possibly some cross-class clustering where images
  share early-layer visual processing (blur, texture, shape type).

In [ ]:
import umap

@torch.no_grad()
def extract_z_embeddings(backbone, meta_enc, root, device, n=2000, split='test', seed=0):
    """Extract z for n images from the val set."""
    backbone.eval(); meta_enc.eval()
    ds = CIFAR10(root=root, train=(split=='train'), transform=val_tf, download=True)
    rng_u = np.random.default_rng(seed)
    indices = rng_u.choice(len(ds), n, replace=False)
    zs, lbls = [], []
    for idx in tqdm(indices, desc='Extracting z', leave=False):
        x, y = ds[idx]
        _, traj = backbone(x.unsqueeze(0).to(device))
        z = meta_enc(traj).cpu().numpy()
        zs.append(z[0])
        lbls.append(y)
    return np.stack(zs), np.array(lbls)


N_UMAP = 2000
print(f'Extracting z embeddings for UMAP (n={N_UMAP})...')
z_nb01_umap, lbl_nb01_umap = extract_z_embeddings(backbone_nb01, meta_enc_nb01, DATA_DIR, DEVICE, n=N_UMAP)
z_nb02_umap, lbl_nb02_umap = extract_z_embeddings(backbone_nb02, meta_enc_nb02, DATA_DIR, DEVICE, n=N_UMAP)

print('Running UMAP...')
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine',
                    random_state=SEED, verbose=False)
umap_nb01 = reducer.fit_transform(z_nb01_umap)
reducer2  = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine',
                      random_state=SEED, verbose=False)
umap_nb02 = reducer2.fit_transform(z_nb02_umap)

CIFAR_CLASSES = ['plane', 'car', 'bird', 'cat', 'deer',
                 'dog',   'frog', 'horse', 'ship', 'truck']
cmap = plt.cm.get_cmap('tab10', 10)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (emb, lbl), title in zip(
        axes,
        [(umap_nb01, lbl_nb01_umap), (umap_nb02, lbl_nb02_umap)],
        ['nb01: class-label CTLS (collapsed)', 'nb02: traj-sim CTLS (Step 5)']):
    for c in range(10):
        mask = lbl == c
        ax.scatter(emb[mask, 0], emb[mask, 1], s=8, alpha=0.5,
                   color=cmap(c), label=CIFAR_CLASSES[c])
    ax.set_title(title, fontsize=12)
    ax.axis('off')

handles = [plt.Line2D([0],[0], marker='o', color='w',
                       markerfacecolor=cmap(c), markersize=8,
                       label=CIFAR_CLASSES[c]) for c in range(10)]
axes[1].legend(handles=handles, loc='lower right', fontsize=8, ncol=2)
plt.suptitle('UMAP of z-space: nb01 (class collapse) vs nb02 (circuit structure)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print()
print('Interpretation guide:')
print('  nb01: tight class blobs → class-level collapse. Each class = one cluster.')
print('  nb02 (expected): class blobs with visible sub-clusters = within-class circuit diversity.')
print('  nb02 (expected): occasional cross-class proximity where visual processing is shared.')

### 9a — Within-Class Sub-Cluster Analysis

Quantify within-class sub-cluster structure using the silhouette score within each class.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans

# Circuit silhouette (class-level): how well do class labels separate z-space?
circuit_sil_nb01 = silhouette_score(z_nb01_umap, lbl_nb01_umap, metric='cosine')
circuit_sil_nb02 = silhouette_score(z_nb02_umap, lbl_nb02_umap, metric='cosine')
print(f'Circuit silhouette (class-level separation):')
print(f'  nb01: {circuit_sil_nb01:.3f}')
print(f'  nb02: {circuit_sil_nb02:.3f}')
print(f'  Note: nb02 may score lower — trajectory-sim pairs allow z to encode sub-class structure,')
print(f'  trading class separation for instance-level circuit fidelity. This is expected.')
print()

# Within-class sub-cluster analysis: for each class, fit K=4 k-means and measure
# intra-cluster silhouette. High silhouette = strong within-class sub-structure.
K_SUBCLUSTERS = 4
within_class_sil_nb01 = []
within_class_sil_nb02 = []

for c in range(10):
    for (z_all, lbls_all, sil_list) in [
        (z_nb01_umap, lbl_nb01_umap, within_class_sil_nb01),
        (z_nb02_umap, lbl_nb02_umap, within_class_sil_nb02),
    ]:
        z_c = z_all[lbls_all == c]
        if len(z_c) < K_SUBCLUSTERS + 1:
            sil_list.append(float('nan'))
            continue
        km = KMeans(n_clusters=K_SUBCLUSTERS, random_state=SEED, n_init=5)
        sub_lbls = km.fit_predict(z_c)
        if len(np.unique(sub_lbls)) > 1:
            sil = silhouette_score(z_c, sub_lbls, metric='euclidean')
        else:
            sil = 0.0
        sil_list.append(sil)

print(f'Within-class sub-cluster silhouette (K={K_SUBCLUSTERS} clusters per class):')
print(f'  Class  | nb01 sil | nb02 sil | Δ')
print(f'  -------+----------+----------+------')
for c in range(10):
    d = within_class_sil_nb02[c] - within_class_sil_nb01[c]
    print(f'  {CIFAR_CLASSES[c]:7s} | {within_class_sil_nb01[c]:.3f}    | {within_class_sil_nb02[c]:.3f}    | {d:+.3f}')
print()
print(f'  Mean nb01: {np.nanmean(within_class_sil_nb01):.3f}')
print(f'  Mean nb02: {np.nanmean(within_class_sil_nb02):.3f}')
print(f'  Higher within-class sil in nb02 → stronger within-class sub-cluster structure.')

## Section 10 — Non-Triviality Summary

Consolidated pass/fail table for all five non-triviality tests. A result is considered
non-trivial if it would not occur by chance or by simply replicating the nb01 approach.

In [ ]:
print('='*70)
print('STEP 5 NON-TRIVIALITY SUMMARY')
print('='*70)
print()

# Gather all test results
tests = [
    ('T1', 'Class-level collapse broken',
     'within-image gap over same-class > nb01',
     t1_pass,
     f'nb01 gap={gap_nb01_collapse:+.3f}  nb02 gap={gap_nb02_collapse:+.3f}'),

    ('T2', 'z adds over h₈',
     'Δρ(z,labels) - Δρ(h₈,labels) > 0.02 in nb02 vs nb01',
     t2_pass,
     f'nb01 z-h₈ Δρ={z_advantage_nb01:+.3f}  nb02={z_advantage_nb02:+.3f}'),

    ('T3', 'Early-layer gaps > 0',
     'More layers 1-4 with gap>0.05 in nb02 than nb01',
     t3_pass,
     f'nb01: {n_early_gaps_nb01}/4 layers  nb02: {n_early_gaps_nb02}/4 layers'),

    ('T4', 'Cross-class positive pairs present',
     'Mean cross-class rate in top-20 neighbors > 0.05',
     cross_rate > 0.05,
     f'Cross-class rate = {cross_rate:.3f}'),

    ('T5', 'Per-layer ρ profile flatter',
     'Early-layer mean ρ improved by >0.05 in nb02',
     t5_pass,
     f'Early ρ: nb01={early_rho_nb01:.3f}  nb02={early_rho_nb02:.3f}'),
]

n_pass = 0
for tid, name, criterion, passed, evidence in tests:
    status = 'PASS ✓' if passed else 'FAIL ✗'
    n_pass += int(passed)
    print(f'  {tid}: {name}')
    print(f'       Criterion: {criterion}')
    print(f'       Evidence:  {evidence}')
    print(f'       Result:    {status}')
    print()

print(f'  Total: {n_pass}/5 tests passed')
print()
if n_pass >= 4:
    print('  CONCLUSION: Step 5 finds non-trivial results. Trajectory-similarity pairs')
    print('  produce measurably different circuit organization than class-label pairs.')
elif n_pass >= 2:
    print('  CONCLUSION: Partial success. Some metrics improved but collapse not fully')
    print('  broken. Consider Phase 2 with fresh top-k index recomputed mid-training,')
    print('  or a longer Phase 2 (>70 epochs), or lower k (k<20) for sharper pairs.')
else:
    print('  CONCLUSION: Step 5 did not produce meaningful changes in circuit organization.')
    print('  Likely cause: Phase 1 class-label training already collapsed z so deeply that')
    print('  the trajectory embeddings used for Phase 2 pairs are still class-dominated.')
    print('  Next: try computing top-k pairs from a baseline ResNet18 (no CTLS loss).')
print('='*70)

## Section 11 — Summary Results Table

In [ ]:
rho_z_traj_nb01_mu, rho_z_traj_nb01_lo, rho_z_traj_nb01_hi = boot_results.get(
    'ρ(z-sim, traj-sim) nb01', (float('nan'), float('nan'), float('nan')))
rho_z_traj_nb02_mu, rho_z_traj_nb02_lo, rho_z_traj_nb02_hi = boot_results.get(
    'ρ(z-sim, traj-sim) nb02', (float('nan'), float('nan'), float('nan')))
rho_z_lbl_nb01 = boot_results.get('ρ(z-sim, labels) nb01', (float('nan'),)*3)[0]
rho_z_lbl_nb02 = boot_results.get('ρ(z-sim, labels) nb02', (float('nan'),)*3)[0]
rho_h8_lbl_nb01 = boot_results.get('ρ(h₈-sim, labels)  nb01', (float('nan'),)*3)[0]
rho_h8_lbl_nb02 = boot_results.get('ρ(h₈-sim, labels)  nb02', (float('nan'),)*3)[0]

print('STEP 5 RESULTS SUMMARY')
print()
print(f'  Metric                               | nb01 (class-label) | nb02 (traj-sim)')
print(f'  -------------------------------------+--------------------+----------------')
print(f'  Val accuracy                         | {nb01_acc:.4f}           | {acc_nb02:.4f}')
print(f'  Circuit silhouette                   | {circuit_sil_nb01:.3f}              | {circuit_sil_nb02:.3f}')
print(f'  Proxy ρ(z-sim, mean-traj-sim)       | {rho_z_traj_nb01_mu:.3f} [{rho_z_traj_nb01_lo:.3f},{rho_z_traj_nb01_hi:.3f}] | {rho_z_traj_nb02_mu:.3f} [{rho_z_traj_nb02_lo:.3f},{rho_z_traj_nb02_hi:.3f}]')
print(f'  ρ(z-sim, labels)                    | {rho_z_lbl_nb01:.3f}              | {rho_z_lbl_nb02:.3f}')
print(f'  ρ(h₈-sim, labels)                   | {rho_h8_lbl_nb01:.3f}              | {rho_h8_lbl_nb02:.3f}')
print(f'  z advantage over h₈ (Δρ)           | {z_advantage_nb01:+.3f}             | {z_advantage_nb02:+.3f}')
print(f'  Early-layer ρ mean (layers 1-4)     | {early_rho_nb01:.3f}              | {early_rho_nb02:.3f}')
print(f'  Early layers with gap>0.05 (of 4)   | {n_early_gaps_nb01}                  | {n_early_gaps_nb02}')
print(f'  Within-image gap (collapse test)    | {gap_nb01_collapse:+.3f}             | {gap_nb02_collapse:+.3f}')
print(f'  Within-img z-sim                    | {wi_nb01.mean():.3f}              | {wi_nb02.mean():.3f}')
print(f'  Same-class z-sim                    | {sa_nb01.mean():.3f}              | {sa_nb02.mean():.3f}')
print(f'  Diff-class z-sim                    | {di_nb01.mean():.3f}              | {di_nb02.mean():.3f}')
print(f'  Cross-class pair rate (top-20)      | —                  | {cross_rate:.3f}')
print(f'  Non-triviality tests passed         | —                  | {n_pass}/5')
print()
print(f'  NB01 checkpoint: {NB01_CKPT}')
print(f'  NB02 checkpoint: {NB02_BEST}')

## Appendix — Interpretation Notes

### What each outcome means

**If T4 shows low cross-class rate (< 0.05):**
The Phase 1 trajectory embeddings are already class-dominated (because Phase 1 used class-label pairs).
The Step 5 training signal is effectively the same as nb01. Fix: compute top-k pairs from a
standard ResNet18 baseline (no CTLS loss), where class labels do not drive z organization.

**If T1/T2/T3 fail but T4 passes:**
Cross-class pairs are present but training didn't sufficiently reorganize the backbone.
Possible causes: (1) λ=1.0 from the start of Phase 2 is too aggressive; (2) 70 epochs is
too short for the backbone to re-organize; (3) class labels in the task loss counteract the
traj-sim signal. Fix: try Phase 2 with reduced task loss weight, or train from scratch without
Phase 1 pre-conditioning.

**If all 5 tests pass:**
Step 5 is validated. The trajectory-similarity pair definition breaks class-level collapse and
produces multi-layer circuit organization. Proceed to Step 6 (SSL extension) or the uniform
weighting ablation.